# 07 - Neural Net (PyTorch MLP)

A different model family - a deep MLP with a learned `Race` embedding, standardized numeric inputs, BatchNorm + dropout - to **decorrelate** from the GBDT base learners. Same 10-fold `StratifiedGroupKFold` and the same per-fold target encodings as `03_Baselines`. Saves `data/train_oof_nn_v{VER}.npy` and `data/test_pred_nn_v{VER}.npy` in the standard `(N_SPLITS, len(test))` layout so it drops straight into `04`/`05`/`06`.

In [ ]:
VER = 1

Imports & seed.

In [ ]:
import os, copy, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn

np.random.seed(42); torch.manual_seed(42)
print('torch', torch.__version__, '| cpu threads', torch.get_num_threads())

Load engineered features (same parquet as the rest of the pipeline).

In [ ]:
train = pd.read_parquet('data/train_features.parquet')
test  = pd.read_parquet('data/test_features.parquet')
print('Train:', train.shape, '| Test:', test.shape)

Features, CV split, and per-fold target encodings - identical to `03_Baselines`.

In [ ]:
TARGET = 'PitNextLap'
# Same convention as 03_Baselines: drop id/target/raw-categoricals; Race gets an
# embedding (the trees use it natively, the linear model drops it).
DROP = ['id', TARGET, 'Driver', 'Compound']
FEATURES_NUM = [c for c in train.columns if c not in DROP and c != 'Race']
TE_COLS = ['driver_pit_rate', 'driver_compound_pit_rate', 'race_compound_pit_rate']

N_SPLITS = 10
RANDOM_STATE = 42
y = train[TARGET].astype(int).values
groups = train.groupby(['Race', 'Year']).ngroup().values
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

def fold_rates(tr):
    s = pd.Series(y[tr])
    gmean = s.mean()
    drv = s.groupby(train['Driver'].iloc[tr].values).mean()
    dc  = s.groupby([train['Driver'].iloc[tr].values, train['Compound'].iloc[tr].values]).mean()
    rc  = s.groupby([train['Race'].iloc[tr].values, train['Compound'].iloc[tr].values]).mean()
    return gmean, drv, dc, rc

def add_te(base_df, src_df, rates):
    gmean, drv, dc, rc = rates
    out = base_df.copy()
    out['driver_pit_rate'] = src_df['Driver'].map(drv).fillna(gmean).values
    out['driver_compound_pit_rate'] = dc.reindex(
        pd.MultiIndex.from_arrays([src_df['Driver'], src_df['Compound']])).fillna(gmean).values
    out['race_compound_pit_rate'] = rc.reindex(
        pd.MultiIndex.from_arrays([src_df['Race'], src_df['Compound']])).fillna(gmean).values
    return out

print(f'numeric features: {len(FEATURES_NUM)} (+{len(TE_COLS)} per-fold TEs) + Race embedding')

Integer-encode `Race` for an embedding (0 = unseen race).

In [ ]:
# Integer-encode Race for an embedding. Index 0 is reserved for unseen races.
race_cats = train['Race'].astype('category').cat.categories
race_to_idx = {c: i + 1 for i, c in enumerate(race_cats)}
N_RACE  = len(race_to_idx) + 1
EMB_DIM = int(min(16, max(4, (N_RACE + 1) // 2)))
train_ridx = train['Race'].map(race_to_idx).fillna(0).astype('int64').values
test_ridx  = test['Race'].map(race_to_idx).fillna(0).astype('int64').values
print(f'races: {len(race_to_idx)} known (+OOV) -> embedding dim {EMB_DIM}')

Hyperparameters.

In [ ]:
MAX_EPOCHS, PATIENCE, BATCH = 40, 6, 4096
LR, WD = 1e-3, 1e-5
SEEDS_NN = [42, 17, 2024]      # averaged per fold to tame NN variance
N_FOLDS_RUN = N_SPLITS         # run every fold
DO_SAVE = True

MLP definition + per-fold/seed training with early stopping on validation AUC.

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_num, n_race, emb_dim, hidden=(256, 128, 64), drop=(0.30, 0.20, 0.10)):
        super().__init__()
        self.emb = nn.Embedding(n_race, emb_dim)
        layers, prev = [], n_num + emb_dim
        for h, p in zip(hidden, drop):
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(p)]
            prev = h
        self.body = nn.Sequential(*layers)
        self.head = nn.Linear(prev, 1)

    def forward(self, xnum, xrace):
        x = torch.cat([xnum, self.emb(xrace)], dim=1)
        return self.head(self.body(x)).squeeze(1)

def _predict(model, Xs, ridx, bs=16384):
    model.eval()
    out = np.empty(len(Xs), dtype=np.float64)
    with torch.no_grad():
        for i in range(0, len(Xs), bs):
            xb = torch.from_numpy(Xs[i:i + bs]); rb = torch.from_numpy(ridx[i:i + bs])
            out[i:i + bs] = torch.sigmoid(model(xb, rb)).numpy()
    return out

def train_fold_seed(Xs_tr, r_tr, y_tr, Xs_va, r_va, y_va, n_num, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = MLP(n_num, N_RACE, EMB_DIM)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    lossf = nn.BCEWithLogitsLoss()
    yf = y_tr.astype(np.float32)
    best_auc, best_state, bad = -1.0, None, 0
    for epoch in range(MAX_EPOCHS):
        model.train()
        perm = np.random.permutation(len(y_tr))
        for i in range(0, len(perm), BATCH):
            idx = perm[i:i + BATCH]
            if len(idx) < 2:          # BatchNorm needs >1 row
                continue
            opt.zero_grad()
            out = model(torch.from_numpy(Xs_tr[idx]), torch.from_numpy(r_tr[idx]))
            loss = lossf(out, torch.from_numpy(yf[idx]))
            loss.backward(); opt.step()
        auc = roc_auc_score(y_va, _predict(model, Xs_va, r_va))
        if auc > best_auc + 1e-5:
            best_auc, best_state, bad = auc, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model, best_auc

Cross-validated training. Averages the seeds per fold; writes OOF + per-fold test preds.

In [ ]:
oof_nn  = np.zeros(len(train))
pred_nn = np.zeros((N_SPLITS, len(test)))

for fold, (tr, va) in enumerate(cv.split(train, y, groups)):
    if fold >= N_FOLDS_RUN:
        break
    t0 = time.time()
    rates = fold_rates(tr)
    Xn_tr = add_te(train[FEATURES_NUM].iloc[tr], train.iloc[tr], rates)
    Xn_va = add_te(train[FEATURES_NUM].iloc[va], train.iloc[va], rates)
    Xn_te = add_te(test[FEATURES_NUM],           test,           rates)

    # finite-ize, then fill from TRAIN means only (no leakage)
    inf = [np.inf, -np.inf]
    Xn_tr, Xn_va, Xn_te = Xn_tr.replace(inf, np.nan), Xn_va.replace(inf, np.nan), Xn_te.replace(inf, np.nan)
    fill = Xn_tr.mean()
    Xn_tr, Xn_va, Xn_te = Xn_tr.fillna(fill), Xn_va.fillna(fill), Xn_te.fillna(fill)

    scaler = StandardScaler()
    Xs_tr = np.clip(scaler.fit_transform(Xn_tr), -10, 10).astype(np.float32)
    Xs_va = np.clip(scaler.transform(Xn_va),     -10, 10).astype(np.float32)
    Xs_te = np.clip(scaler.transform(Xn_te),     -10, 10).astype(np.float32)
    n_num = Xs_tr.shape[1]
    r_tr, r_va = train_ridx[tr], train_ridx[va]

    fva = np.zeros(len(va)); fte = np.zeros(len(test)); saucs = []
    for seed in SEEDS_NN:
        model, bauc = train_fold_seed(Xs_tr, r_tr, y[tr], Xs_va, r_va, y[va], n_num, seed)
        fva += _predict(model, Xs_va, r_va)    / len(SEEDS_NN)
        fte += _predict(model, Xs_te, test_ridx) / len(SEEDS_NN)
        saucs.append(bauc)
    oof_nn[va]    = fva
    pred_nn[fold] = fte
    print(f'Fold {fold + 1} AUC: {roc_auc_score(y[va], fva):.5f}  '
          f'(seed best {np.mean(saucs):.5f}+/-{np.std(saucs):.5f})  {time.time() - t0:.0f}s')

if N_FOLDS_RUN == N_SPLITS:
    print('-' * 40)
    print(f'Neural Net, CV OOF AUC: {roc_auc_score(y, oof_nn):.5f}')

Save artifacts, then a diversity diagnostic: correlation to each GBDT and the hill-climb gain from adding the NN.

In [ ]:
if DO_SAVE:
    np.save(f'data/train_oof_nn_v{VER}.npy', oof_nn)
    np.save(f'data/test_pred_nn_v{VER}.npy', pred_nn)
    print(f'saved data/train_oof_nn_v{VER}.npy {oof_nn.shape} and '
          f'data/test_pred_nn_v{VER}.npy {pred_nn.shape}')

    # --- diversity diagnostic: is the NN decorrelated, and does it help the blend? ---
    from data.utils import hill_climb_ensemble
    def neg_auc(yt, yp):
        return 1.0 - roc_auc_score(yt, yp)

    base_names = ['linear', 'xgb', 'lgb', 'cb', 'stack_target', 'stack_feature']
    print(f'\nNN OOF AUC: {roc_auc_score(y, oof_nn):.5f}')
    print('Pearson corr(nn, model)  [lower = more diverse]:')
    for k in base_names:
        ko = np.load(f'data/train_oof_{k}_v{VER}.npy')
        print(f'  nn vs {k:14s} {np.corrcoef(oof_nn, ko)[0, 1]:.4f}')

    def run_hc(names):
        ol, tl = [], []
        for k in names:
            if k == 'nn':
                ol.append(oof_nn); tl.append(pred_nn.mean(0))
            else:
                ol.append(np.load(f'data/train_oof_{k}_v{VER}.npy'))
                tl.append(np.load(f'data/test_pred_{k}_v{VER}.npy').mean(0))
        res = hill_climb_ensemble(oofs=ol, test_preds=tl, names=names, y_true=y,
                                  metric=neg_auc, max_number_models=None,
                                  tolerance=1e-6, use_negative_weights=False, verbose=False)
        return roc_auc_score(y, res['oof_pred']), res

    auc_wo, _     = run_hc(base_names)
    auc_w,  res_w = run_hc(base_names + ['nn'])
    print(f'\nhill-climb OOF AUC  without nn: {auc_wo:.5f}')
    print(f'hill-climb OOF AUC  with    nn: {auc_w:.5f}')
    print(f'delta from adding nn         : {auc_w - auc_wo:+.5f}')
    print('weights (with nn):')
    alln = base_names + ['nn']
    for m, w in zip(res_w['used_models'], res_w['weights']):
        print(f'  {alln[m]:14s} {w:.3f}')

AUC-error comparison incl. the NN.

In [ ]:
# AUC-error comparison incl. NN (mirrors 06's chart).
from sklearn.metrics import roc_auc_score
names = ['linear', 'xgb', 'lgb', 'cb', 'stack_target', 'stack_feature', 'pseudo_full', 'nn']
aucs = []
for k in names:
    o = oof_nn if k == 'nn' else np.load(f'data/train_oof_{k}_v{VER}.npy')
    aucs.append(roc_auc_score(y, o))
plt.figure(figsize=(9, 4))
plt.bar(names, [1 - a for a in aucs])
plt.ylabel('1 - AUC  [lower is better]'); plt.yscale('log')
plt.title('Model AUC-error comparison (incl. NN)')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()